# Парсинг кинопоиска

Описание:
Парсинг популярных актеров кинопоиска
Итогом парсинга является таблица, следующего вида:


### Импорты

In [ ]:
# pip install bs4

In [ ]:
import json
import requests
import datetime
import pandas as pd
from bs4 import BeautifulSoup
import re
import tqdm
import tqdm._tqdm
import time
import random
d=datetime.datetime.now().strftime('%Y-%m-%d')

### Период дней для парсинга

In [ ]:
what_days=0

In [ ]:
if what_days==0:
    ## С какой даты мы считаем
    days=[datetime.date(2026,1,1)]
    ### Все дни подряд до вчерашнего добавляем в список
    while days[-1]!=datetime.date.today():
        days+=[days[-1]+datetime.timedelta(days=1)]
    days=[x.strftime("%Y-%m-%d") for x in days]
if what_days==1:
    days=['2022-03-31']
# days

In [ ]:
ps=['','page/2/','page/3/','page/4/','page/5/']

In [ ]:
# Формируем список из сочетаний дат и страниц
page_sources=[]
for day in tqdm.tqdm(days):
    for p in ps:
        page_sources+= [["https://www.kinopoisk.ru/popular/names/day/"+day+"/"+p,day]]
# page_sources

### Парсим

In [ ]:
d0=[]
d1=[]
d2=[]
d3=[]
d4=[]

In [ ]:
## Проходим в цикле все ссылки, удаляем ссылку, по которой удалось пробиться
while len(page_sources)>0:
    try:
        random.shuffle(page_sources)
        pss=page_sources.copy()
        for page_source in tqdm.tqdm(page_sources):
            response = requests.get(page_source[0])
            soup = BeautifulSoup(response.text, "html.parser")
            sfa=soup.findAll('div', {'class':'el'})
            if len(sfa)>0:
                pss.remove(page_source)
                for a in sfa:
                    d0.append(page_source[1])
                    if 'персона удалена' in (str(a)):
                        d1.append(str(a).split('>')[3].replace('</span',''))
                        d2.append('персона удалена')
                        d3.append('')
                        d4.append('')
                    else:
                        d1.append(str(a).split('>')[3].replace('</a',''))
                        d2.append(str(a).split('>')[6].replace('</a',''))
                        b3=str(a).split('>')[8]
                        if '</i' in b3:
                            d3.append(str(a).split('>')[8].replace('</i',''))
                            d4.append('eng')
                        else:
                            d3.append('')
                            d4.append('rus')
        page_sources=pss.copy()
        print(len(page_sources))
    except:
        continue

In [12]:
dictt = {"Дата":d0,
        "Num":d1,
        "Актёр":d2,
        "Актёр_eng":d3,
        "Страна":d4}
df = pd.DataFrame(dictt)
df

,Дата,Num,Актёр,Актёр_eng,Страна
0,2026-01-12,201,Роман Панков,,rus
1,2026-01-12,202,Юлия Снигирь,,rus
2,2026-01-12,203,Колин Фаррелл,Colin Farrell,eng
3,2026-01-12,204,Линда Лапиньш,,rus
4,2026-01-12,205,Роман Курцын,,rus
...,...,...,...,...,...
53395,2026-01-18,396,Крис Хемсворт,Chris Hemsworth,eng
53396,2026-01-18,397,Сергей Марин,,rus
53397,2026-01-18,398,Лев Зулькарнаев,,rus
53398,2026-01-18,399,Маколей Калкин,Macaulay Culkin,eng


In [13]:
df.to_csv('Актёры_itg.csv', index= False)